In [10]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [11]:
from src.config.config_manager import ConfigManager
from src.ghcn_daily.ghcn_data_handler import GHCNDataHandler
from src.ghcn_daily.data_fetch import DataFetcher,CurrentYearMonthDataFetcher
from src.ghcn_daily.data_processing import WeatherDataProcessor
import numpy as np

In [12]:
ghcn = GHCNDataHandler()
config = ConfigManager(config_directory='/workspaces/BlizzardX/src/config')
config.load_config('settings.json')
data_fetcher = DataFetcher(config_file="settings.json",data_type='dataframe')
latest_data = CurrentYearMonthDataFetcher(config_file="settings.json",data_type='dataframe')

In [13]:
stations= ghcn.get_station_data(config.get('settings.json', 'data_sources.stations'))
inventory = ghcn.get_inventory_data(config.get('settings.json', 'data_sources.inventory'))

In [15]:
inventory.columns

Index(['ID', 'LATITUDE', 'LONGITUDE', 'ELEMENT', 'FIRSTYEAR', 'LASTYEAR'], dtype='object')

In [16]:
s_state_list=stations[stations['STATE']=='NH']['ID'].tolist()
s_live_list=inventory[(inventory['ID'].isin(s_state_list)) & (inventory['LASTYEAR']>2024) & (inventory['FIRSTYEAR']<2015)]['ID'].unique().tolist()

In [17]:
data= await data_fetcher.save_data(s_live_list)

Fetching Data: 100%|██████████████████████████████████████████████████| 8/8 [00:06<00:00,  1.21it/s]


CPU usage is high! Decreasing workers to 4
CPU usage is high! Decreasing workers to 2
CPU usage is stable. Increasing workers to 4
CPU usage is stable. Increasing workers to 6


In [18]:
flag_columns = [col for col in data.columns if 'FLAG' in col]
data= data.drop(columns=flag_columns)
data.replace(-9999.0, np.nan, inplace=True)
weather_variables = ['TMAX', 'TMIN', 'SNOW', 'SNWD', 'PRCP']

In [19]:
processor=WeatherDataProcessor(data,weather_variables)

In [20]:
df=processor.process_data()

In [21]:
import pandas as pd
def list_stations_with_less_than_5_percent_missing(df):
    stations = df["ID"].unique()
    stations_with_less_than_5_percent_missing = []
    for station in stations:
        station_data = df[df["ID"] == station].copy()
        station_data.loc[:, 'DATE'] = pd.to_datetime(station_data['DATE'])
        station_data.sort_index(inplace=True)
        columns_to_check = ['TMIN', 'TMAX', 'SNOW', 'SNWD', 'PRCP']
        missing_percentage = station_data[columns_to_check].isnull().mean() * 100
        if (missing_percentage < 10).all():
            stations_with_less_than_5_percent_missing.append(station)
    return stations_with_less_than_5_percent_missing
list = list_stations_with_less_than_5_percent_missing(df)
len(list)

11

In [22]:
df=df[df['ID'].isin(list)]

In [23]:
import pandas as pd
df = pd.merge(df, stations, on='ID', how='left')
df = df[['DATE','ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season', 'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD']]

In [24]:
df.to_csv('/workspaces/BlizzardX/Data/processed_data.csv', index=False)

In [2]:
import pandas as pd
df=pd.read_csv('/workspaces/BlizzardX/Data/processed_data.csv')

In [26]:
df.isnull().sum()

DATE            0
ID              0
LATITUDE        0
LONGITUDE       0
ELEVATION       0
NAME            0
Season          0
TMIN         1966
TMAX         2055
PRCP         1576
SNOW         3406
SNWD         5848
dtype: int64

In [27]:
from src.ghcn_daily.data_processing import  WeatherDataCleaner
cleaner = WeatherDataCleaner(df)

In [28]:
df=cleaner.clean_all(alpha=0.6)

In [33]:
df.to_csv('/workspaces/BlizzardX/Data/cleaned_data.csv', index=False)

In [32]:
df.isnull().sum()

DATE          0
ID            0
LATITUDE      0
LONGITUDE     0
ELEVATION     0
NAME          0
Season        0
TMIN         18
TMAX         18
PRCP          0
SNOW          0
SNWD          0
dtype: int64

In [31]:
df[df.isnull().any(axis=1)]

,DATE,ID,LATITUDE,LONGITUDE,ELEVATION,NAME,Season,TMIN,TMAX,PRCP,SNOW,SNWD
0,1960-07-01,USC00271647,44.8611,-71.5392,341.4,NH COLEBROOK 3SW,Summer,NaN,NaN,3.48,0.00,0.00
20204,2019-07-10,USC00271647,44.8611,-71.5392,341.4,NH COLEBROOK 3SW,Summer,NaN,25.34,0.00,0.00,0.00
22226,2025-01-21,USC00271647,44.8611,-71.5392,341.4,NH COLEBROOK 3SW,Winter,-27.80,NaN,0.00,0.00,0.00
22311,2008-07-20,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Summer,NaN,NaN,0.30,0.00,0.00
27370,2022-05-27,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,NaN,23.30,0.00,0.00,0.00
28307,2024-12-19,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Winter,-1.70,NaN,6.90,0.00,0.00
28530,1973-11-29,USC00273626,43.0211,-70.8258,29.0,NH GREENLAND,Fall,NaN,NaN,0.00,0.00,0.00
46621,2024-06-14,USC00273626,43.0211,-70.8258,29.0,NH GREENLAND,Summer,NaN,NaN,0.00,0.00,0.00
47044,2011-10-31,USC00274304,42.8089,-72.0053,313.9,NH JAFFREY SILVER RCH AIRPARK,Fall,-5.60,NaN,0.00,709.33,617.40
47045,2011-11-02,USC00274304,42.8089,-72.0053,313.9,NH JAFFREY SILVER RCH AIRPARK,Fall,NaN,11.10,0.00,620.67,548.80


In [40]:
from src.Model.feature_engineering import FeatureEngineering
fe=FeatureEngineering(df)

In [41]:
df=fe.apply_all_features()

AttributeError: 'FeatureEngineering' object has no attribute 'groupby'